# Prototype Implementation of OpenAI API

In [16]:
using OpenAI
using LinearAlgebra, BenchmarkTools, MatrixMarket

In [4]:
secret_key = ENV["OPENAI_API_KEY"];
model = "gpt-5-mini"

"gpt-5-mini"

In [11]:
function make_agent(key, model, role_description)
    return create_chat(key,
                       model,
                       [Dict("role" => "developer", "content" => role_description)])
end

make_agent (generic function with 1 method)

In [12]:
a = make_agent(secret_key, model, "You are a texting agent.")

OpenAIResponse{JSON3.Object{Vector{UInt8}, Vector{UInt64}}}(200, {
                   "id": "chatcmpl-CMeYZqBjsdITpBjIOJis6zSmzbF3R",
               "object": "chat.completion",
              "created": 1759515151,
                "model": "gpt-5-mini-2025-08-07",
              "choices": [
                           {
                                      "index": 0,
                                    "message": {
                                                         "role": "assistant",
                                                      "content": "Hi! Let’s explain photosynthesis in a fun, simple way for a 7‑year‑old.\n\nThink of a leaf as a little kitchen. Plants are chefs who make their own food using sunlight.\n\nWhat plants need:\n- Sunlight — like the stove’s heat.\n- Water — plants drink it through their roots.\n- Air (carbon dioxide) — plants breathe it in from around them.\n\nHow it works (easy steps):\n1. The leaf catches sunlight with its green stuff (called chlorop

In [15]:
prompt =  "Say \"this is a test\""
r = create_chat(
    secret_key,
    model,
    [Dict("role" => "user", "content"=> prompt)]
  )
println(r.response[:choices][begin][:message][:content])

this is a test


In [16]:
r

OpenAIResponse{JSON3.Object{Vector{UInt8}, Vector{UInt64}}}(200, {
                   "id": "chatcmpl-CMefUgWXzml1Qv5ppIqHbN3gRQVCA",
               "object": "chat.completion",
              "created": 1759515580,
                "model": "gpt-5-mini-2025-08-07",
              "choices": [
                           {
                                      "index": 0,
                                    "message": {
                                                         "role": "assistant",
                                                      "content": "this is a test",
                                                      "refusal": nothing,
                                                  "annotations": []
                                               },
                              "finish_reason": "stop"
                           }
                         ],
                "usage": {
                                        "prompt_tokens": 22,
                            

In [17]:
prompt =  "You are an expert software developer. Implement a sorting algorithm in Julia. Only return the function. Do not return extra text."
r = create_chat(
    secret_key,
    model,
    [Dict("role" => "user", "content"=> prompt)]
  )
println(r.response[:choices][begin][:message][:content])

function quicksort!(a::AbstractVector{T}, lo::Int=1, hi::Int=length(a)) where T
    if hi <= lo
        return a
    end
    # choose a random pivot and place it at lo
    p = rand(lo:hi)
    pivot = a[p]
    a[lo], a[p] = a[p], a[lo]
    lt = lo
    i = lo + 1
    gt = hi
    while i <= gt
        if a[i] < pivot
            lt += 1
            a[lt], a[i] = a[i], a[lt]
            i += 1
        elseif pivot < a[i]
            a[i], a[gt] = a[gt], a[i]
            gt -= 1
        else
            i += 1
        end
    end
    a[lo], a[lt] = a[lt], a[lo]
    quicksort!(a, lo, lt - 1)
    quicksort!(a, gt + 1, hi)
    return a
end


In [18]:
gen_code = r.response[:choices][begin][:message][:content] 

"function quicksort!(a::AbstractVector{T}, lo::Int=1, hi::Int=length(a)) where T\n    if hi <= lo\n        return a\n    end\n    # choose a random pivot and place it at lo\n    p = rand(lo:hi)\n    pivot = a[p]\n    a[lo], a[p] = a[p], a[lo]\n    lt = lo\n    i = lo + 1\n    gt = " ⋯ 97 bytes ⋯ "a[lt]\n            i += 1\n        elseif pivot < a[i]\n            a[i], a[gt] = a[gt], a[i]\n            gt -= 1\n        else\n            i += 1\n        end\n    end\n    a[lo], a[lt] = a[lt], a[lo]\n    quicksort!(a, lo, lt - 1)\n    quicksort!(a, gt + 1, hi)\n    return a\nend"

In [19]:
@eval(gen_code)

"function quicksort!(a::AbstractVector{T}, lo::Int=1, hi::Int=length(a)) where T\n    if hi <= lo\n        return a\n    end\n    # choose a random pivot and place it at lo\n    p = rand(lo:hi)\n    pivot = a[p]\n    a[lo], a[p] = a[p], a[lo]\n    lt = lo\n    i = lo + 1\n    gt = " ⋯ 97 bytes ⋯ "a[lt]\n            i += 1\n        elseif pivot < a[i]\n            a[i], a[gt] = a[gt], a[i]\n            gt -= 1\n        else\n            i += 1\n        end\n    end\n    a[lo], a[lt] = a[lt], a[lo]\n    quicksort!(a, lo, lt - 1)\n    quicksort!(a, gt + 1, hi)\n    return a\nend"

In [21]:
eval(Meta.parse(gen_code))

quicksort! (generic function with 3 methods)

In [23]:
a = rand(1:10, 10)

10-element Vector{Int64}:
  3
  4
 10
  7
  4
  4
  5
  3
  6
  4

In [24]:
quicksort!(a)

10-element Vector{Int64}:
  3
  3
  4
  4
  4
  4
  5
  6
  7
 10

In [26]:
function test_sorted(a)
    return sum(1 .- (a .== sort(a))) == 0
end

test_sorted (generic function with 1 method)

In [28]:
test_sorted(rand(1:10, 10))

false

In [47]:
using CodeTracking
b = @code_string test_sorted(a)

In [49]:
println(b)

nothing


In [50]:
function myfunction(x, y)
result = x + y
return result * 2
end

function_string = @code_string myfunction(1, 2)

In [52]:
println(function_string)

nothing


In [56]:
write("test_sorted.jl", "function test_sorted(a)
    return sum(1 .- (a .== sort(a))) == 0
end")

69

In [5]:
content = "begin\n" * read("test_sorted.jl", String)* "\nend"

"begin\nfunction test_sorted(a)\n    return sum(1 .- (a .== Base.sort(a))) == 0\nend\nfunction checker(sorting_fn)\n    n = 1000\n    a = rand(1:n, n)\n    a = sorting_fn(a)\n    return test_sorted(a)\nend\nend"

In [15]:
proposed_fn(x) = x

proposed_fn (generic function with 1 method)

In [16]:
function generate_code(prompt, secret_key, model, checker_fn_str, max_iters = 3)
    eval(Meta.parse(checker_fn_str))
    dev_prompt = "You are an expert software developer. The user will ask you to generate a function and use the following code the check if your solution is correct. Here is the code: \n" * checker_fn_str * "\nOnly return the function. Make sure the function name is proposed_fn. Do not return extra text. "
    chat_history = [Dict("role" => "developer", "content"=> dev_prompt),
                    Dict("role" => "user", "content"=> prompt),]
    r = create_chat(
        secret_key,
        model,
        [Dict("role" => "developer", "content"=> dev_prompt),
        Dict("role" => "user", "content"=> prompt)]
    )
    # good_prompt =  "You are an expert software developer. " * prompt * " Only return the function. Do not return extra text. "
    iters = 0
    gen_code = r.response[:choices][begin][:message][:content] 
    eval(Meta.parse(gen_code))
    push!(chat_history, Dict("role" => "assistant", "content" => gen_code))
    
    println(gen_code)
    while iters < max_iters & ~checker(proposed_fn)
        next_prompt = "I ran the code using a checker function. It failed. I need you to fix it."
        push!(chat_history, Dict("role" => "user", "content" => next_prompt))
        r = create_chat(secret_key,
                        model,
                        chat_history)
        gen_code = r.response[:choices][begin][:message][:content] 
        proposed_fn = eval(Meta.parse(gen_code))
        push!(chat_history, Dict("role" => "assistant", "content" => gen_code))
        iters += 1
        println(iters)
        println(gen_code)
    end
    return gen_code, chat_history, iters == max_iters
end

generate_code (generic function with 2 methods)

In [17]:
res= generate_code("Implement a merge sort algorithm in Julia.", secret_key, model, content)

function proposed_fn(a::AbstractVector{T}) where T
    n = length(a)
    if n <= 1
        return copy(a)
    end
    mid = n ÷ 2
    left = proposed_fn(a[1:mid])
    right = proposed_fn(a[mid+1:end])
    ln = length(left)
    rn = length(right)
    res = similar(a)
    i = 1
    j = 1
    k = 1
    while i <= ln && j <= rn
        if left[i] <= right[j]
            res[k] = left[i]
            i += 1
        else
            res[k] = right[j]
            j += 1
        end
        k += 1
    end
    while i <= ln
        res[k] = left[i]
        i += 1
        k += 1
    end
    while j <= rn
        res[k] = right[j]
        j += 1
        k += 1
    end
    return res
end
1
function proposed_fn(a::Vector{T}) where T
    n = length(a)
    if n <= 1
        return copy(a)
    end
    src = copy(a)
    dest = similar(src)
    function sort!(src, dest, l, r)
        if l == r
            dest[l] = src[l]
            return
        end
        m = (l + r) ÷ 2
        sort!(dest, src, l, 

("function proposed_fn(a::Vector{T}) where T\n    n = length(a)\n    if n <= 1\n        return copy(a)\n    end\n    src = copy(a)\n    dest = similar(src)\n    function sort!(src, dest, l, r)\n        if l == r\n            dest[l] = src[l]\n            return\n        end\n        m = (l + r) ÷ 2\n        sort!(dest, src, l, m)\n        sort!(dest, src, m+1, r)\n        i = l\n        j = m + 1\n        k = l\n        while i <= m && j <= r\n            if src[i] <= src[j]\n                dest[k] = src[i]\n                i += 1\n            else\n                dest[k] = src[j]\n                j += 1\n            end\n            k += 1\n        end\n        while i <= m\n            dest[k] = src[i]\n            i += 1\n            k += 1\n        end\n        while j <= r\n            dest[k] = src[j]\n            j += 1\n            k += 1\n        end\n    end\n    sort!(src, dest, 1, n)\n    return dest\nend", [Dict("role" => "developer", "content" => "You are an expert soft

In [9]:
proposed_fn

proposed_fn (generic function with 1 method)

In [10]:
b = rand(1:100, 100)

100-element Vector{Int64}:
 51
 63
 17
 34
 81
 86
 76
 29
 90
 67
 74
 49
 81
  ⋮
 91
 23
 88
 34
 41
 39
  1
 79
 61
 62
 34
 28

In [11]:
proposed_fn(b)

100-element Vector{Int64}:
           0
           0
           0
           0
           0
           0
           0
           0
           0
           0
           0
           0
           1
           ⋮
  4766115712
  4766115712
  4766117472
  4766121008
  4766122112
 13251678496
 13251678608
 13254530640
 13254756464
 13254756496
 13254756816
 13254756848

In [12]:
checker

checker (generic function with 1 method)

In [13]:
checker(proposed_fn)

true

In [18]:
hist = [Dict("role" => "developer", "content" => "You are an expert software developer. The user will ask you to generate a function and use the following code the check if your solution is correct. Here is the code: \nbegin\nfunction test_sorted(a)\n    return sum(1 .- (a .== Base.sort(a))) == 0\nend\nfunction checker(sorting_fn)\n    n = 1000\n    a = rand(1:n, n)\n    a = sorting_fn(a)\n    return test_sorted(a)\nend\nend\nOnly return the function. Make sure the function name is proposed_fn. Do not return extra text. "), Dict("role" => "user", "content" => "Implement a merge sort algorithm in Julia."), Dict("role" => "assistant", "content" => "function proposed_fn(a::AbstractVector{T}) where T\n    n = length(a)\n    if n <= 1\n        return copy(a)\n    end\n    mid = n ÷ 2\n    left = proposed_fn(a[1:mid])\n    right = proposed_fn(a[mid+1:end])\n    ln = length(left)\n    rn = length(right)\n    res = similar(a)\n    i = 1\n    j = 1\n    k = 1\n    while i <= ln && j <= rn\n        if left[i] <= right[j]\n            res[k] = left[i]\n            i += 1\n        else\n            res[k] = right[j]\n            j += 1\n        end\n        k += 1\n    end\n    while i <= ln\n        res[k] = left[i]\n        i += 1\n        k += 1\n    end\n    while j <= rn\n        res[k] = right[j]\n        j += 1\n        k += 1\n    end\n    return res\nend"), Dict("role" => "user", "content" => "I ran the code using a checker function. It failed. I need you to fix it."), Dict("role" => "assistant", "content" => "function proposed_fn(a::Vector{T}) where T\n    n = length(a)\n    if n <= 1\n        return copy(a)\n    end\n    src = copy(a)\n    dest = similar(src)\n    function sort!(src, dest, l, r)\n        if l == r\n            dest[l] = src[l]\n            return\n        end\n        m = (l + r) ÷ 2\n        sort!(dest, src, l, m)\n        sort!(dest, src, m+1, r)\n        i = l\n        j = m + 1\n        k = l\n        while i <= m && j <= r\n            if src[i] <= src[j]\n                dest[k] = src[i]\n                i += 1\n            else\n                dest[k] = src[j]\n                j += 1\n            end\n            k += 1\n        end\n        while i <= m\n            dest[k] = src[i]\n            i += 1\n            k += 1\n        end\n        while j <= r\n            dest[k] = src[j]\n            j += 1\n            k += 1\n        end\n    end\n    sort!(src, dest, 1, n)\n    return dest\nend")]

5-element Vector{Dict{String, String}}:
 Dict("role" => "developer", "content" => "You are an expert software developer. The user will ask you to generate a function and use the following code the check if your solution is correct. Here is the code: \nbegin\nfunction test_sorted(a)\n    return sum(1 .- (a .== Base.sort(a))) == 0\nend\nfunction checker(sorting_fn)\n    n = 1000\n    a = rand(1:n, n)\n    a = sorting_fn(a)\n    return test_sorted(a)\nend\nend\nOnly return the function. Make sure the function name is proposed_fn. Do not return extra text. ")
 Dict("role" => "user", "content" => "Implement a merge sort algorithm in Julia.")
 Dict("role" => "assistant", "content" => "function proposed_fn(a::AbstractVector{T}) where T\n    n = length(a)\n    if n <= 1\n        return copy(a)\n    end\n    mid = n ÷ 2\n    left = proposed_fn(a[1:mid])\n    right = proposed_fn(a[mid+1:end])\n    ln = length(left)\n    rn = length(right)\n    res = similar(a)\n    i = 1\n    j = 1\n    k = 1\n 

In [24]:
hist = hist[2:end]

5-element Vector{Dict{String, String}}:
 Dict("role" => "user", "content" => "Implement a merge sort algorithm in Julia.")
 Dict("role" => "assistant", "content" => "function proposed_fn(a::AbstractVector{T}) where T\n    n = length(a)\n    if n <= 1\n        return copy(a)\n    end\n    mid = n ÷ 2\n    left = proposed_fn(a[1:mid])\n    right = proposed_fn(a[mid+1:end])\n    ln = length(left)\n    rn = length(right)\n    res = similar(a)\n    i = 1\n    j = 1\n    k = 1\n    while i <= ln && j <= rn\n        if left[i] <= right[j]\n            res[k] = left[i]\n            i += 1\n        else\n            res[k] = right[j]\n            j += 1\n        end\n        k += 1\n    end\n    while i <= ln\n        res[k] = left[i]\n        i += 1\n        k += 1\n    end\n    while j <= rn\n        res[k] = right[j]\n        j += 1\n        k += 1\n    end\n    return res\nend")
 Dict("role" => "user", "content" => "I ran the code using a checker function. It failed. I need you to fix it.")

In [25]:
r = create_chat(secret_key, model,hist)

OpenAIResponse{JSON3.Object{Vector{UInt8}, Vector{UInt64}}}(200, {
                   "id": "chatcmpl-CMgCK3U4puR5BSenBwXcKSwCR4HBV",
               "object": "chat.completion",
              "created": 1759521460,
                "model": "gpt-5-mini-2025-08-07",
              "choices": [
                           {
                                      "index": 0,
                                    "message": {
                                                         "role": "assistant",
                                                      "content": "Summary of the conversation\n\n- You asked for a merge sort implementation in Julia.\n- I first returned a straightforward recursive implementation (proposed_fn) that:\n  - Took AbstractVector{T}, split the array into left/right using slicing,\n  - Recursively sorted each half,\n  - Merged into a new array allocated with similar(a),\n  - Returned the merged result.\n\n- You reported the code failed a checker and asked for a fix.\n\n

In [27]:
println(r.response[:choices][begin][:message][:content] )

Summary of the conversation

- You asked for a merge sort implementation in Julia.
- I first returned a straightforward recursive implementation (proposed_fn) that:
  - Took AbstractVector{T}, split the array into left/right using slicing,
  - Recursively sorted each half,
  - Merged into a new array allocated with similar(a),
  - Returned the merged result.

- You reported the code failed a checker and asked for a fix.

- I then provided a corrected implementation (proposed_fn) that:
  - Accepts Vector{T} and makes a copy src = copy(a) plus a dest = similar(src),
  - Uses an inner recursive sort!(src, dest, l, r) that alternates the roles of src and dest on recursive calls so merging always reads from src and writes to dest,
  - Handles singleton ranges by copying the element from src to dest,
  - Merges ranges using 1-based indexing (Julia style) and returns dest when finished.

- Key differences between the two versions:
  - First version used slicing and allocated new arrays at eve

## Evaluator test

In [45]:
eval_str = "begin\n" * read("test_performance.jl", String)* "\nend"

"begin\nusing LinearAlgebra, BenchmarkTools\nproj_dir = @__DIR__ \nmatrix_dir = proj_dir * \"/Matrices/\"\ntest_matrix_names = matrix_dir .* [\"af23560.mtx\", \"airfoil_2d.mtx\", \"cage10.mtx\"]\ntest_matrices = mmread.(test_matrix_names)\nbase_prompt(rel_errs, speedups) = \"Here are th" ⋯ 394 bytes ⋯ "m(x_exact))\n        base_runtime = @btimed \$A \\ \$b\n        alg_runtime = @btimed \$proposed_fn(\$A, \$b)\n        push!(speedups, base_runtime.time/alg_runtime.time)\n        println(\"done\")\n    end\n    return mean(rel_errors) < tol, base_prompt(rel_errors, speedups)\nend\nend"

In [46]:
eval(Meta.parse(eval_str))

evaluator (generic function with 2 methods)

In [47]:
base(A, b) = A \ b

base (generic function with 1 method)

In [48]:
evaluator(base)

done
done
done


(true, "Here are the errors compared to built-in linear solver: \n[0.0, 0.0, 0.0]\nand here are the speed-up ratio compared to built-in solver:\n[1.3778461263386281, 1.6913896371946668, 0.8103621948687199].")